# 01: Data Acquisition & Profiling

**Objective**: Load, validate, and profile all datasets needed for the
Fashion Trend Intelligence Pipeline.

| Dataset | Type | Purpose |
|---------|------|---------|
| H&M Articles | Structured | Product attributes (type, color, section) |
| H&M Transactions | Structured | Sell-through data (dates, prices, volumes) |
| Women's Clothing Reviews | Unstructured | Customer sentiment & style signals |

---

## 1.1 Environment Setup

In [ ]:
import sys
import os
import logging
import warnings

warnings.filterwarnings("ignore")

# ── Auto-detect environment: Colab / Local ──
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/FashionTrendAnalyzer"
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.system("pip install -q peft pytrends hdbscan umap-learn pydantic sentence-transformers")
    print(f"Colab: project at {PROJECT_ROOT}")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.config import (
    DATA_RAW, DATA_PROCESSED, PROJECT_ROOT as PROJ_ROOT,
    hm_config, reviews_config,
)
from src.data_loader import (
    load_hm_articles, load_hm_transactions, load_reviews,
    validate_schema, log_data_quality, save_processed,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid", palette="muted")

print(f"Project root: {PROJ_ROOT}")
print(f"Raw data dir: {DATA_RAW}")
print(f"Processed dir: {DATA_PROCESSED}")

In [ ]:
# Verify files exist
expected_files = [
    DATA_RAW / hm_config.articles_file,
    DATA_RAW / hm_config.transactions_file,
    DATA_RAW / reviews_config.file,
]
for f in expected_files:
    status = "OK" if f.exists() else "MISSING — download and place in data/raw/"
    print(f"  [{status}] {f.name}")

---
## 1.3 Load & Validate  H&M Articles

In [ ]:
articles = load_hm_articles()
print(f"Shape: {articles.shape}")
articles.head(3)

In [ ]:
# Key attribute distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

articles["product_type_name"].value_counts().head(20).plot.barh(
    ax=axes[0, 0], title="Top 20 Product Types"
)
articles["colour_group_name"].value_counts().head(15).plot.barh(
    ax=axes[0, 1], title="Top 15 Colour Groups"
)
articles["section_name"].value_counts().plot.barh(
    ax=axes[1, 0], title="Sections"
)
articles["garment_group_name"].value_counts().head(15).plot.barh(
    ax=axes[1, 1], title="Garment Groups"
)

for ax in axes.flat:
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Null analysis
null_pct = (articles.isnull().sum() / len(articles) * 100).sort_values(ascending=False)
print("Columns with nulls:")
print(null_pct[null_pct > 0].round(2))

---
## 1.4 Load & Validate  H&M Transactions


In [ ]:
SAMPLE_FRAC = 0.10

transactions = load_hm_transactions(sample_frac=SAMPLE_FRAC)
print(f"Shape: {transactions.shape}")
print(f"Date range: {transactions['t_dat'].min()} to {transactions['t_dat'].max()}")
transactions.head(3)

In [ ]:
# Transaction volume over time
daily_volume = transactions.groupby("t_dat").size().rename("transactions")

fig = px.line(
    daily_volume.reset_index(),
    x="t_dat", y="transactions",
    title="Daily Transaction Volume Over Time",
    labels={"t_dat": "Date", "transactions": "# Transactions"},
    template="plotly_white",
)
fig.update_layout(height=350, width=900)
fig.show()

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

transactions["price"].clip(upper=transactions["price"].quantile(0.99)).hist(
    bins=50, ax=axes[0], edgecolor="white"
)
axes[0].set_title("Price Distribution (clipped at 99th percentile)")
axes[0].set_xlabel("Price")

transactions.groupby(transactions["t_dat"].dt.day_name())["price"].mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
).plot.bar(ax=axes[1], title="Average Transaction Price by Day of Week")

plt.tight_layout()
plt.show()

---
## 1.5 Load & Validate Women's Clothing Reviews

In [ ]:
reviews = load_reviews()
print(f"Shape: {reviews.shape}")
reviews.head(3)

In [ ]:
# Review distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

reviews["Rating"].value_counts().sort_index().plot.bar(
    ax=axes[0, 0], title="Rating Distribution", color="steelblue"
)

reviews["Department Name"].value_counts().plot.barh(
    ax=axes[0, 1], title="Reviews by Department"
)

reviews["Class Name"].value_counts().head(15).plot.barh(
    ax=axes[1, 0], title="Top 15 Product Classes"
)

reviews["Review Text"].str.len().hist(
    bins=50, ax=axes[1, 1], edgecolor="white"
)
axes[1, 1].set_title("Review Length Distribution")
axes[1, 1].set_xlabel("Characters")

plt.tight_layout()
plt.show()

In [ ]:
# Recommendation rate by department
rec_rate = (
    reviews
    .groupby("Department Name")["Recommended IND"]
    .mean()
    .sort_values(ascending=False)
)
fig = px.bar(
    rec_rate.reset_index(),
    x="Department Name", y="Recommended IND",
    title="Recommendation Rate by Department",
    labels={"Recommended IND": "Recommendation Rate"},
    template="plotly_white",
    text_auto=".1%",
)
fig.update_layout(height=350, width=700)
fig.show()

---
## 1.6 Data Quality Summary

In [ ]:
quality_summary = pd.DataFrame({
    "Dataset": ["H&M Articles", "H&M Transactions", "Reviews"],
    "Rows": [len(articles), len(transactions), len(reviews)],
    "Columns": [articles.shape[1], transactions.shape[1], reviews.shape[1]],
    "Null Columns (>0%)": [
        (articles.isnull().any()).sum(),
        (transactions.isnull().any()).sum(),
        (reviews.isnull().any()).sum(),
    ],
    "Memory (MB)": [
        articles.memory_usage(deep=True).sum() / 1e6,
        transactions.memory_usage(deep=True).sum() / 1e6,
        reviews.memory_usage(deep=True).sum() / 1e6,
    ],
})
quality_summary["Memory (MB)"] = quality_summary["Memory (MB)"].round(1)
quality_summary

---
## 1.7 Save Cleaned Data for Downstream Processes

In [ ]:
save_processed(articles, "articles_clean")
save_processed(transactions, "transactions_sample")
save_processed(reviews, "reviews_clean")

print("Saved to data/processed/:")
for f in DATA_PROCESSED.glob("*.parquet"):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} — {size_mb:.1f} MB")